In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import random
from pathlib import Path
import sys
import json


from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report

import torch.nn as nn
import torch

d:\Python\Learning\NLP_Classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
project_root = Path().resolve().parent

# добавляем в sys.path
sys.path.append(str(project_root))

from core.graf import *
from core.func import *
from core.nn_models import *

In [4]:
# ── Конфиг ──────────────────────────────────────────────────────────────────
DATA_PATH           = '../data/News_Category_Dataset_v3.json'
MODEL_NAME          = 'roberta-base' # 'cointegrated/rubert-tiny2'
MODEL_DIR           = f'models/{MODEL_NAME.split("/")[-1]}' 
SAVE_PATH           = f'news_classifier_{MODEL_NAME.split("/")[-1]}'
MAX_LEN             = 500
BATCH_SIZE          = 32
EPOCHS              = 100
ACCUM_STEPS         = 4
WARMUP_FRAC         = 0.1
LR                  = 2e-5
SEED                = 42
DEVICE              = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("GPU: ", torch.cuda.get_device_name())
    print("VRAM:", round(torch.cuda.get_device_properties().total_memory / 1e9, 1),  'Gb')

random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

GPU:  NVIDIA GeForce RTX 5070
VRAM: 12.8 Gb


In [5]:
with open(DATA_PATH, 'r') as file:
    data = pd.DataFrame([json.loads(line) for line in file])

data.sample(5)

,link,headline,category,short_description,authors,date
128310,https://www.huffingtonpost.com/entry/what-if-w...,What If We Were All Family Generation Changers?,IMPACT,"What if, in doing so, we won't just create new...","Matt Murrie, ContributorEdupreneur, Cofounder/...",2014-06-20
139983,https://www.huffingtonpost.comhttp://www.washi...,Firestorm At AOL Over Employee Benefit Cuts,BUSINESS,It should have been a glorious week for AOL ch...,,2014-02-08
42339,https://www.huffingtonpost.com/entry/time-runs...,Dakota Access Protesters Arrested As Deadline ...,POLITICS,A few protesters who refused to leave remained...,"Michael McLaughlin & Josh Morgan, The Huffingt...",2017-02-22
131494,https://www.huffingtonpost.com/entry/one-glimp...,One Glimpse Of These Baby Kit Foxes And You'll...,GREEN,,,2014-05-14
163649,https://www.huffingtonpost.com/entry/mens-swea...,"Mens' Sweat Pheromone, Androstadienone, Influe...",SCIENCE,Scientists didn't know if humans played that g...,Melissa Cronin,2013-06-02


In [6]:
preprocesed_data = data.copy()
preprocesed_data.drop_duplicates(inplace=True)
preprocesed_data["headline"] = preprocesed_data["headline"].str.replace("’", "'")
preprocesed_data["short_description"] = preprocesed_data["short_description"].str.replace("’", "'")
preprocesed_data["input_text"] = preprocesed_data["headline"] + " [SEP] " + preprocesed_data["authors"] + " [SEP] " + preprocesed_data["short_description"]
y = preprocesed_data[["category"]]
X = preprocesed_data["input_text"]

In [7]:
train_X, val_X, train_y, val_y = train_test_split(X, y, test_size=0.4, random_state=SEED, shuffle=True, stratify=y)
val_X, test_X, val_y, test_y = train_test_split(val_X, val_y, test_size=0.5, random_state=SEED, shuffle=True, stratify=val_y)

In [8]:
# ── Dataset с претокенизацией ─────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
le = OneHotEncoder() # LabelEncoder()

def batch_tokenization(tokenizer, texts, desc: str = "Токенизация", batch_size: int = 5000):
    all_ids, all_masks = [], []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        tokens = tokenizer(
            texts[i : i+batch_size],
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors='pt'
        )
        all_ids.append(tokens["input_ids"])
        all_masks.append(tokens["attention_mask"])
    return torch.cat(all_ids), torch.cat(all_masks)

In [9]:
train_ids, train_masks = batch_tokenization(tokenizer=tokenizer, texts=train_X.to_list())
train_labels = le.fit_transform(train_y).toarray()
train_labels

Токенизация: 100%|██████████| 26/26 [00:19<00:00,  1.30it/s]


array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(125708, 42))

In [10]:
counts = train_labels.sum(axis=0)

pos_weight = torch.tensor([(len(train_labels) - c) / c for c in counts], dtype=torch.float).to(DEVICE)
print('pos_weight:', pos_weight)

pos_weight: tensor([137.9039, 155.5479,  44.7120,  33.9675, 182.2478,  37.7988,  57.8245,
        193.8961,  60.1420, 205.4171,  11.0676, 144.1593, 148.4744,  32.0463,
        148.8307,  78.9161,  30.3018,  47.4985,  59.1474, 184.4100,  70.1823,
        118.2676,  22.8309,  51.9743,   4.8849,  32.0116,  80.3118,  93.9456,
         40.2699,  91.9793,  20.3535,  98.9269,  98.7683,  56.1920,  20.1630,
        151.1889,  56.3485,  74.4550,  10.6775,  57.6598,  62.5210,  80.2592],
       device='cuda:0')


In [11]:
train_ds = TextDataset(train_ids, train_masks, train_labels)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

In [12]:
val_ids, val_masks = batch_tokenization(tokenizer=tokenizer, texts=val_X.to_list())
val_labels = le.transform(val_y).toarray()

val_ds = TextDataset(val_ids, val_masks, val_labels)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

Токенизация: 100%|██████████| 9/9 [00:06<00:00,  1.36it/s]


In [13]:
lstm_model = LSTM_Classifier(num_classes=len(le.categories_[0]),
                             vocab_size=len(tokenizer.get_vocab()),
                             embedding_dim=256,
                             lstm_units=64,
                             num_layers=1,
                             )

In [ ]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = AdamW(lstm_model.parameters(), lr=0.001, weight_decay=0.01)
lstm_model.to(DEVICE)

training(
    model=lstm_model,
    train_data_loader=train_dl,
    val_data_loader=val_dl,
    optimizer=optimizer,
    epoches=EPOCHS,
    criterion=criterion,
    tokenizator=le,
    accum_steps=ACCUM_STEPS,
    warmup_frac=WARMUP_FRAC,
    device=DEVICE,
    save_path="LSTM_news_classifier"
    )

Epoch 1/100: 100%|██████████| 3929/3929 [00:27<00:00, 144.96it/s, loss=0.9198]


Epoch 1 | train_loss=0.8182 | val_loss=0.8262 | val_f1_mean=0.1263 |


,ARTS,ARTS & CULTURE,BLACK VOICES,BUSINESS,COLLEGE,COMEDY,CRIME,CULTURE & ARTS,DIVORCE,EDUCATION,...,TECH,THE WORLDPOST,TRAVEL,U.S. NEWS,WEDDINGS,WEIRD NEWS,WELLNESS,WOMEN,WORLD NEWS,WORLDPOST
f1,0.069712,0.054299,0.0765,0.116769,0.036239,0.151257,0.088747,0.028227,0.099049,0.052889,...,0.04929,0.1293,0.195194,0.060027,0.105036,0.080088,0.448853,0.089077,0.120475,0.13425


  → Сохранена лучшая модель (mean F1=0.1263)


Epoch 2/100: 100%|██████████| 3929/3929 [00:26<00:00, 148.90it/s, loss=0.6490]


...

,ARTS,ARTS & CULTURE,BLACK VOICES,BUSINESS,COLLEGE,COMEDY,CRIME,CULTURE & ARTS,DIVORCE,EDUCATION,...,TECH,THE WORLDPOST,TRAVEL,U.S. NEWS,WEDDINGS,WEIRD NEWS,WELLNESS,WOMEN,WORLD NEWS,WORLDPOST
f1,0.681672,0.629126,0.561305,0.540426,0.428894,0.63234,0.513016,0.497674,0.776224,0.517483,...,0.569412,0.531561,0.805241,0.27,0.762155,0.438144,0.825384,0.458228,0.468795,0.604525


Epoch 98/100: 100%|██████████| 3929/3929 [00:26<00:00, 150.02it/s, loss=0.0001]


Epoch 98 | train_loss=0.0009 | val_loss=3.0467 | val_f1_mean=0.6160 |


,ARTS,ARTS & CULTURE,BLACK VOICES,BUSINESS,COLLEGE,COMEDY,CRIME,CULTURE & ARTS,DIVORCE,EDUCATION,...,TECH,THE WORLDPOST,TRAVEL,U.S. NEWS,WEDDINGS,WEIRD NEWS,WELLNESS,WOMEN,WORLD NEWS,WORLDPOST
f1,0.684628,0.629126,0.558398,0.540609,0.431461,0.62995,0.513583,0.5,0.77707,0.523585,...,0.571765,0.529015,0.804005,0.272414,0.762409,0.437554,0.825259,0.459057,0.462342,0.610701


,ARTS,ARTS & CULTURE,BLACK VOICES,BUSINESS,COLLEGE,COMEDY,CRIME,CULTURE & ARTS,DIVORCE,EDUCATION,...,TECH,THE WORLDPOST,TRAVEL,U.S. NEWS,WEDDINGS,WEIRD NEWS,WELLNESS,WOMEN,WORLD NEWS,WORLDPOST
f1,0.684628,0.629126,0.558398,0.540609,0.431461,0.62995,0.513583,0.5,0.77707,0.523585,...,0.571765,0.529015,0.804005,0.272414,0.762409,0.437554,0.825259,0.459057,0.462342,0.610701


In [ ]:
rnn_model = RNN_Text_Classifier(
    num_classes=len(le.categories_[0]),
    vocab_size=len(tokenizer.get_vocab()),
    embedding_dim=256,
    hidden_size=64,
)

In [ ]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = AdamW(rnn_model.parameters(), lr=0.001, weight_decay=0.01)
rnn_model.to(DEVICE)

training(
    model=rnn_model,
    train_data_loader=train_dl,
    val_data_loader=val_dl,
    optimizer=optimizer,
    epoches=EPOCHS,
    criterion=criterion,
    tokenizator=le,
    accum_steps=ACCUM_STEPS,
    warmup_frac=WARMUP_FRAC,
    device=DEVICE,
    save_path="RNN_news_classifier"
    )

Epoch 1/100: 100%|██████████| 3929/3929 [00:20<00:00, 191.61it/s, loss=1.0024]


Epoch 1 | train_loss=1.3570 | val_loss=1.3269 | val_f1_mean=0.0573 |


,ARTS,ARTS & CULTURE,BLACK VOICES,BUSINESS,COLLEGE,COMEDY,CRIME,CULTURE & ARTS,DIVORCE,EDUCATION,...,TECH,THE WORLDPOST,TRAVEL,U.S. NEWS,WEDDINGS,WEIRD NEWS,WELLNESS,WOMEN,WORLD NEWS,WORLDPOST
f1,0.052103,0.017836,0.043781,0.057645,0.019585,0.061354,0.046257,0.019593,0.046925,0.009441,...,0.028508,0.042915,0.080482,0.022516,0.05595,0.028506,0.198272,0.034512,0.041638,0.117872


  → Сохранена лучшая модель (mean F1=0.0573)


Epoch 2/100: 100%|██████████| 3929/3929 [00:19<00:00, 198.97it/s, loss=1.1704]


...

Epoch 100/100: 100%|██████████| 3929/3929 [00:20<00:00, 194.12it/s, loss=0.3046]


Epoch 100 | train_loss=0.3514 | val_loss=2.3460 | val_f1_mean=0.1527 |


,ARTS,ARTS & CULTURE,BLACK VOICES,BUSINESS,COLLEGE,COMEDY,CRIME,CULTURE & ARTS,DIVORCE,EDUCATION,...,TECH,THE WORLDPOST,TRAVEL,U.S. NEWS,WEDDINGS,WEIRD NEWS,WELLNESS,WOMEN,WORLD NEWS,WORLDPOST
f1,0.105189,0.094096,0.09191,0.148685,0.055405,0.122301,0.156801,0.023121,0.133233,0.055491,...,0.106055,0.206915,0.215204,0.060909,0.135058,0.147186,0.365173,0.080461,0.203287,0.211867


  → Сохранена лучшая модель (mean F1=0.1527)


,ARTS,ARTS & CULTURE,BLACK VOICES,BUSINESS,COLLEGE,COMEDY,CRIME,CULTURE & ARTS,DIVORCE,EDUCATION,...,TECH,THE WORLDPOST,TRAVEL,U.S. NEWS,WEDDINGS,WEIRD NEWS,WELLNESS,WOMEN,WORLD NEWS,WORLDPOST
f1,0.105189,0.094096,0.09191,0.148685,0.055405,0.122301,0.156801,0.023121,0.133233,0.055491,...,0.106055,0.206915,0.215204,0.060909,0.135058,0.147186,0.365173,0.080461,0.203287,0.211867


In [44]:
from huggingface_hub import snapshot_download

print("📥 Скачивание модели...")
model_path = snapshot_download(
    repo_id=MODEL_NAME,
    local_dir=MODEL_DIR,
)

📥 Скачивание модели...


Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 777.98it/s]


In [45]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, num_labels=len(le.categories_[0]), local_files_only=True)
model.to(DEVICE)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 9983.18it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: models/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
  

In [47]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
model.to(DEVICE)

training(
    model=model,
    train_data_loader=train_dl,
    val_data_loader=val_dl,
    optimizer=optimizer,
    epoches=EPOCHS,
    criterion=criterion,
    tokenizator=le,
    accum_steps=ACCUM_STEPS,
    warmup_frac=WARMUP_FRAC,
    device=DEVICE,
    save_path=SAVE_PATH,
    )

Epoch 1/100: 100%|██████████| 3929/3929 [18:31<00:00,  3.54it/s, loss=0.8248]


Epoch 1 | train_loss=1.2622 | val_loss=0.9502 | val_f1_mean=0.1363 |


,ARTS,ARTS & CULTURE,BLACK VOICES,BUSINESS,COLLEGE,COMEDY,CRIME,CULTURE & ARTS,DIVORCE,EDUCATION,...,TECH,THE WORLDPOST,TRAVEL,U.S. NEWS,WEDDINGS,WEIRD NEWS,WELLNESS,WOMEN,WORLD NEWS,WORLDPOST
f1,0.077277,0.040941,0.0826,0.164804,0.054297,0.107239,0.111209,0.080032,0.126306,0.029454,...,0.063844,0.164963,0.315234,0.031488,0.157666,0.048974,0.38517,0.084747,0.160403,0.112924


  → Сохранена лучшая модель (mean F1=0.1363)


Epoch 2/100: 100%|██████████| 3929/3929 [18:38<00:00,  3.51it/s, loss=0.5479]


...

  → Сохранена лучшая модель (mean F1=0.7462)


Epoch 29/100: 100%|██████████| 3929/3929 [18:25<00:00,  3.55it/s, loss=0.0021]


Epoch 29 | train_loss=0.0140 | val_loss=1.0284 | val_f1_mean=0.7436 |


,ARTS,ARTS & CULTURE,BLACK VOICES,BUSINESS,COLLEGE,COMEDY,CRIME,CULTURE & ARTS,DIVORCE,EDUCATION,...,TECH,THE WORLDPOST,TRAVEL,U.S. NEWS,WEDDINGS,WEIRD NEWS,WELLNESS,WOMEN,WORLD NEWS,WORLDPOST
f1,0.741194,0.719424,0.675835,0.694351,0.641975,0.736888,0.68614,0.778325,0.872752,0.618337,...,0.73774,0.665914,0.906588,0.444772,0.888298,0.606061,0.903252,0.623507,0.576923,0.714516


,ARTS,ARTS & CULTURE,BLACK VOICES,BUSINESS,COLLEGE,COMEDY,CRIME,CULTURE & ARTS,DIVORCE,EDUCATION,...,TECH,THE WORLDPOST,TRAVEL,U.S. NEWS,WEDDINGS,WEIRD NEWS,WELLNESS,WOMEN,WORLD NEWS,WORLDPOST
f1,0.741194,0.719424,0.675835,0.694351,0.641975,0.736888,0.68614,0.778325,0.872752,0.618337,...,0.73774,0.665914,0.906588,0.444772,0.888298,0.606061,0.903252,0.623507,0.576923,0.714516


In [21]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, num_labels=len(le.categories_[0]), local_files_only=True)
model.load_state_dict(torch.load('news_classifier_roberta-base_best.pt', weights_only=True))
model.to(DEVICE)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 10640.92it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: models/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
  

In [22]:
model.eval()

pred_labels, mask_labels = [], []

with torch.no_grad():
    for batch in val_dl:
        log = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE)).logits
        pred_labels.append((torch.sigmoid(log) > 0.5).to('cpu').numpy())
        mask_labels.append(batch['labels'].numpy())
    
    pred_labels = np.vstack(pred_labels)
    mask_labels = np.vstack(mask_labels)
    
    print(classification_report(mask_labels, pred_labels, target_names=le.categories_[0]))

                precision    recall  f1-score   support

          ARTS       0.76      0.80      0.78       302
ARTS & CULTURE       0.79      0.74      0.76       268
  BLACK VOICES       0.68      0.74      0.71       917
      BUSINESS       0.66      0.78      0.72      1198
       COLLEGE       0.63      0.60      0.62       229
        COMEDY       0.74      0.75      0.74      1080
         CRIME       0.63      0.74      0.68       713
CULTURE & ARTS       0.87      0.75      0.81       214
       DIVORCE       0.86      0.91      0.89       685
     EDUCATION       0.63      0.68      0.66       202
 ENTERTAINMENT       0.84      0.88      0.86      3473
   ENVIRONMENT       0.65      0.76      0.70       288
         FIFTY       0.77      0.75      0.76       280
  FOOD & DRINK       0.89      0.93      0.91      1268
     GOOD NEWS       0.71      0.65      0.68       280
         GREEN       0.60      0.74      0.67       524
HEALTHY LIVING       0.75      0.83      0.79  

d:\Python\Learning\NLP_Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [24]:
test_ids, test_masks = batch_tokenization(tokenizer=tokenizer, texts=test_X.to_list())
test_labels = le.transform(test_y).toarray()

test_ds = TextDataset(test_ids, test_masks, test_labels)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

model.eval()

pred_labels, mask_labels = [], []

with torch.no_grad():
    for batch in test_dl:
        log = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE)).logits
        pred_labels.append((torch.sigmoid(log) > 0.5).to('cpu').numpy())
        mask_labels.append(batch['labels'].numpy())
    
    pred_labels = np.vstack(pred_labels)
    mask_labels = np.vstack(mask_labels)
    
    print(classification_report(mask_labels, pred_labels, target_names=le.categories_[0]))

Токенизация: 100%|██████████| 9/9 [00:06<00:00,  1.35it/s]


                precision    recall  f1-score   support

          ARTS       0.76      0.81      0.79       302
ARTS & CULTURE       0.79      0.73      0.76       268
  BLACK VOICES       0.69      0.72      0.71       916
      BUSINESS       0.63      0.80      0.70      1199
       COLLEGE       0.68      0.71      0.70       229
        COMEDY       0.75      0.76      0.76      1080
         CRIME       0.66      0.74      0.70       712
CULTURE & ARTS       0.84      0.77      0.81       215
       DIVORCE       0.86      0.90      0.88       685
     EDUCATION       0.64      0.69      0.66       203
 ENTERTAINMENT       0.84      0.88      0.86      3472
   ENVIRONMENT       0.66      0.81      0.73       289
         FIFTY       0.76      0.78      0.77       280
  FOOD & DRINK       0.90      0.93      0.92      1268
     GOOD NEWS       0.67      0.63      0.65       279
         GREEN       0.60      0.71      0.65       525
HEALTHY LIVING       0.74      0.79      0.77  

d:\Python\Learning\NLP_Classification\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
